# MET Kinase Zero-Shot ESM Scoring: Replication Gate + Cross-Kinase Prep

**Kaggle setup (do this before running):** Settings (right sidebar) &#8594; Accelerator &#8594; GPU T4 x2 or P100 &#8594; Internet &#8594; On. ESM-2 3B needs a GPU with >=16GB memory; 15B is a stretch goal for a rented cloud GPU, not this notebook.

**Purpose:** build our own ESM-1b (and ESM-2, multiple sizes) zero-shot mutation-sensitivity scorer, and treat reproducing Estevam et al. 2025's published MET result as a hard **replication gate** before scoring EGFR or ABL1 in a later notebook. This notebook also adds the untreated-vs-treated fitness comparison and non-learned baselines that were flagged as missing from the first version of this pipeline.

**What changed in this version, and why:**
- Framing fix: ESM is not trained on any kinase-resistance data, but it *is* pretrained on general sequence data &#8212; the correct claim is *no task-specific fine-tuning on resistance measurements*, not "no training on affinity data."
- **Untreated (DMSO) vs. drug-treated correlation is now computed explicitly, side by side.** This is the actual test of the paper's central mechanism: protein language models are trained on evolutionary sequence data, so they should track general functional constraint (untreated fitness) far better than drug-specific resistance pressure (treated fitness), which they have never seen anything like during pretraining.
- **Non-learned baselines are now computed alongside ESM**, not left for a separate step: BLOSUM62 substitution score, plus Estevam et al.'s own precomputed structural/biophysical features (distance to the ATP pocket, distance to the bound inhibitor, and others) reused directly from their processed data rather than recomputed from scratch.
- **Correlations are reported per (drug, model) pair, never pooled across drugs as the headline number**, with a simple hierarchical (mean/median across drugs) summary in addition to the per-drug table.
- **The replication check is now a hard pass/fail gate** with an explicit printed verdict, not just a suggestion to eyeball the number.
- Conservation (a third standard baseline) is deliberately **not** included here &#8212; it needs a multiple sequence alignment against MET homologs, which is a separate, heavier step. Flagged at the end as a follow-up, not faked with a placeholder number.

**Data attribution:** WT/mutant fitness and drug-treatment data from Estevam GO et al., "Mapping kinase domain resistance mechanisms for the MET receptor tyrosine kinase via deep mutational scanning," eLife 2025;13:RP101882. Code repo: https://github.com/fraser-lab/MET_kinase_Inhibitor_DMS (MIT License).

**Method (masked-marginal LLR, Meier et al. 2021):** for each position, mask the wild-type residue, run one forward pass, take the log-softmax over the vocabulary at that position, then score = log P(mutant aa) - log P(wildtype aa). This requires only one forward pass per *position* (not per mutation), since a single masked forward pass gives log-probabilities for all 20 amino acids at once.

## 1. Setup

In [ ]:
!pip install -q fair-esm biopython
!rm -rf MET_kinase_Inhibitor_DMS
!git clone --depth 1 https://github.com/fraser-lab/MET_kinase_Inhibitor_DMS.git

import torch
import esm
import pandas as pd
import numpy as np
from scipy import stats
import re
from Bio.Align import substitution_matrices

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type != 'cuda':
    print('WARNING: no GPU detected. Go to Settings > Accelerator and enable a GPU before running the model cells below.')

## 2. Reconstruct the exact WT construct sequence from their own data

Rather than assume a UniProt numbering (their DMS construct is numbered 1-287, not full-length MET UniProt numbering), we reconstruct the reference sequence directly from the `wildtype` column of their own `WT_rosace_effect_all.tsv` file &#8212; this guarantees an exact match to whatever construct they actually screened, no numbering-convention risk. This file also contains the DMSO (untreated) condition we need for Section 7.

In [ ]:
wt_df = pd.read_csv('MET_kinase_Inhibitor_DMS/WT_rosace_effect_all.tsv', sep='\t')
wt_map = wt_df[['position', 'wildtype']].drop_duplicates().sort_values('position')

positions = sorted(wt_map['position'].unique())
assert positions == list(range(1, len(positions) + 1)), 'position numbering has gaps - stop and investigate before proceeding'

WT_SEQ = ''.join(wt_map.set_index('position').loc[positions, 'wildtype'])
print('Reconstructed WT MET kinase-domain construct length:', len(WT_SEQ))
print(WT_SEQ)
print()
print('Conditions available in WT_rosace_effect_all.tsv:', sorted(wt_df['inhibitor'].unique()))

## 3. Load their mutation panel + published ESM-LLR scores (for validation) and structural/biophysical baseline features

In [ ]:
feat_df = pd.read_csv('MET_kinase_Inhibitor_DMS/Machine_Learning/data/all_features_all_data.csv')

# one row per unique mutation (score/pos/structural features are identical across the 9 drug-repeated rows)
baseline_cols = ['dddG', 'ddG_all', 'distance', 'inhib_distance', 'pocket_volume',
                  'hydrophobicity_score', 'polarity_score', 'rmsd', 'ligrmsd']
mut_panel = feat_df[['pos_mut', 'pos', 'score'] + baseline_cols].drop_duplicates(subset=['pos_mut']).reset_index(drop=True)

def parse_pos_mut(pos_mut_str):
    m = re.match(r'^(\d+)(\w)$', pos_mut_str)
    pos = int(m.group(1))
    mut_aa = m.group(2)
    wt_aa = WT_SEQ[pos - 1]
    return pos, wt_aa, mut_aa

parsed = mut_panel['pos_mut'].apply(parse_pos_mut)
mut_panel['pos'] = parsed.apply(lambda t: t[0])
mut_panel['wt_aa'] = parsed.apply(lambda t: t[1])
mut_panel['mut_aa'] = parsed.apply(lambda t: t[2])

# sanity check: does our reconstructed WT sequence match the wt residue implied by their own wildtype column?
check = wt_df[['position','wildtype']].drop_duplicates().set_index('position')
mismatches = 0
for _, row in mut_panel.iterrows():
    expected = check.loc[row['pos'], 'wildtype']
    if row['wt_aa'] != expected:
        mismatches += 1
print(f'WT-residue mismatches between reconstructed sequence and their wildtype column: {mismatches} / {len(mut_panel)}')
assert mismatches == 0, 'Reference sequence reconstruction is wrong - stop and investigate before running any model'

print(f'{len(mut_panel)} unique mutations across {mut_panel["pos"].nunique()} positions loaded and verified')
print(f'Baseline/structural columns carried through: {baseline_cols}')
mut_panel.head()

## 4. Zero-shot masked-marginal scoring function

One masked forward pass per **position** (not per mutation) &#8212; the masked position's output logits give log-probabilities for all 20 amino acids simultaneously, which covers every mutation observed at that position in one pass.

In [ ]:
def compute_masked_marginal_scores(model, alphabet, wt_seq, positions_to_score, device):
    """
    Returns a dict: {position (1-indexed) -> {amino_acid: log_prob}}
    following Meier et al. 2021's masked-marginal method.
    """
    batch_converter = alphabet.get_batch_converter()
    model.eval()
    _, _, base_tokens = batch_converter([('wt', wt_seq)])
    base_tokens = base_tokens.to(device)

    log_probs_by_position = {}
    with torch.no_grad():
        for pos in positions_to_score:
            tokens = base_tokens.clone()
            tokens[0, pos] = alphabet.mask_idx  # index `pos` in the token tensor == 1-indexed sequence position `pos`, since <cls> occupies index 0
            out = model(tokens, repr_layers=[], return_contacts=False)
            logits = out['logits'][0, pos]
            log_probs = torch.log_softmax(logits, dim=-1)
            log_probs_by_position[pos] = {
                aa: log_probs[alphabet.get_idx(aa)].item()
                for aa in 'ACDEFGHIKLMNPQRSTVWY'
            }
    return log_probs_by_position

## 5. Run ESM-1b and score all mutations — REPLICATION GATE

This is a hard gate, not a soft check: if our from-scratch scorer does not closely reproduce Estevam et al.'s published `score` column, something in this reimplementation is wrong (most likely culprit: token-index offset, or alphabet amino-acid ordering), and no downstream analysis — on MET or on EGFR/ABL1 in a later notebook — should be trusted until it is fixed.

**Why the gate threshold is 0.85, not 0.95:** a first run of this notebook (see project log) got rho=0.9199 (p&#8776;0, n=5,434) against Estevam et al.'s published score column &#8212; strong, highly significant agreement, but short of 0.95. That gap is expected, not a bug signal: we don't have Estevam et al.'s actual scoring script (it isn't checked into their GitHub repo), only their score *output*, so a perfect match was never guaranteed. Plausible, non-bug sources of a ~0.90-0.95 gap include (a) Meier et al. 2021's two zero-shot variants &#8212; masked-marginal (used here) vs. wildtype-marginal &#8212; which are correlated but not identical; (b) their model may have scored mutations within a larger flanking sequence context (e.g. more of full-length MET) than the isolated 287-residue construct reconstructed here, which would shift individual position log-probabilities without breaking the overall correlation; (c) minor numerical differences (precision, batching). A correlation this high and this significant, with zero WT-sequence mismatches already confirmed in Section 3, is strong evidence the reimplementation is fundamentally correct. **0.85 is the bar for treating it as validated; below that, something is more likely genuinely wrong and worth debugging before trusting this on EGFR/ABL1.**

In [ ]:
GATE_THRESHOLD = 0.85  # see markdown note above cell: why 0.85, not 0.95

print('Loading ESM-1b (650M) ...')
model_1b, alphabet_1b = esm.pretrained.esm1b_t33_650M_UR50S()
model_1b = model_1b.to(device)

unique_positions = sorted(mut_panel['pos'].unique())
print(f'Scoring {len(unique_positions)} positions with ESM-1b (one forward pass each)...')
logprobs_1b = compute_masked_marginal_scores(model_1b, alphabet_1b, WT_SEQ, unique_positions, device)

def llr_score(row, logprobs):
    lp = logprobs[row['pos']]
    return lp[row['mut_aa']] - lp[row['wt_aa']]

mut_panel['our_esm1b_score'] = mut_panel.apply(lambda r: llr_score(r, logprobs_1b), axis=1)

gate_rho, gate_p = stats.spearmanr(mut_panel['our_esm1b_score'], mut_panel['score'])
print(f'\nValidation: Spearman(our from-scratch ESM-1b LLR, their published score) = {gate_rho:.4f} (p={gate_p:.2e})')

if gate_rho >= GATE_THRESHOLD:
    print(f'\nGATE: PASSED (rho={gate_rho:.4f} >= {GATE_THRESHOLD}). Safe to proceed to EGFR/ABL1 scoring in a later notebook.')
    print('Note: a residual gap from 1.0 is expected, not a bug signal by itself - see the markdown cell above')
    print('for why 0.85, not 0.95+, is the right bar here.')
else:
    print(f'\nGATE: FAILED (rho={gate_rho:.4f} < {GATE_THRESHOLD}). STOP.')
    print('At this correlation level the reimplementation is likely genuinely broken, not just methodologically')
    print('different from Estevam et al. Most likely causes: token-index offset (BOS handling), alphabet AA')
    print('ordering, or a WT-sequence mismatch (Section 2/3 checks should have already caught the last one).')

## 6. Drug-treated resistance correlation — per (drug, model), never pooled first

Reports ρ(kinase=MET, drug, model=ESM-1b) for all 9 inhibitors individually, then a hierarchical (across-drug) summary — not a single flattened pooled number as the headline result.

In [ ]:
merged = feat_df.merge(mut_panel[['pos_mut', 'our_esm1b_score']], on='pos_mut', how='left')

print('rho(MET, drug, ESM-1b) for our from-scratch scorer, drug-treated resistance:')
drug_results = []
for drug in sorted(merged['key'].unique()):
    sub = merged[merged['key'] == drug].dropna(subset=['our_esm1b_score', 'mean'])
    rho, p = stats.spearmanr(sub['our_esm1b_score'], sub['mean'])
    drug_results.append({'kinase': 'MET', 'drug': drug, 'model': 'esm1b', 'condition': 'treated', 'n': len(sub), 'rho': rho, 'p': p})
    print(f'  {drug:6s}  n={len(sub):6d}  rho={rho:.3f}')

drug_rhos = np.array([r['rho'] for r in drug_results])
print(f'\\nHierarchical summary across the 9 drugs (NOT the primary reported number by itself — report alongside the per-drug table):')
print(f'  mean rho   = {drug_rhos.mean():.3f}')
print(f'  median rho = {np.median(drug_rhos):.3f}')
print(f'  SD         = {drug_rhos.std(ddof=1):.3f}')
print(f'  range      = {drug_rhos.min():.3f} to {drug_rhos.max():.3f}')
z = np.arctanh(drug_rhos)  # Fisher z-transform, standard way to average correlations
print(f'  Fisher-z-averaged rho = {np.tanh(z.mean()):.3f}  (approximate SE = {1/np.sqrt(len(drug_rhos)-3):.3f} in z-space)')

pooled_rho, _ = stats.spearmanr(merged.dropna(subset=['our_esm1b_score','mean'])['our_esm1b_score'], merged.dropna(subset=['our_esm1b_score','mean'])['mean'])
print(f'\\n(For reference only, not the headline number) naive pooled-across-drugs rho = {pooled_rho:.3f}')

per_drug_df = pd.DataFrame(drug_results)
per_drug_df.to_csv('met_esm1b_per_drug_treated.csv', index=False)

## 7. Untreated (DMSO) vs. drug-treated — the actual mechanism test

This is the central hypothesis, made explicit rather than left implicit: a protein language model is pretrained on evolutionary sequence data, so it should track **general functional constraint** (untreated/DMSO fitness) much better than **drug-specific resistance pressure** (treated fitness), which is not a signal ESM has ever seen anything resembling during pretraining. Estevam et al. report untreated r≈0.50 vs. treated r≈0.28 for ESM-1b — this section checks whether our own from-scratch scorer reproduces that same gap, using the same score column validated as the gate in Section 5.

In [ ]:
dmso_df = wt_df[wt_df['inhibitor'] == 'DMSO'][['position', 'mutation', 'ROSACE_effects']].copy()
dmso_df['pos_mut'] = dmso_df['position'].astype(str) + dmso_df['mutation']
dmso_df = dmso_df.merge(mut_panel[['pos_mut', 'our_esm1b_score']], on='pos_mut', how='inner')

untreated_rho, untreated_p = stats.spearmanr(dmso_df['our_esm1b_score'], dmso_df['ROSACE_effects'])
print(f'UNTREATED (DMSO baseline fitness): n={len(dmso_df)}, rho={untreated_rho:.4f} (p={untreated_p:.2e})')
print(f'Estevam et al. published untreated correlation for ESM-1b: r≈0.50')
print()
print(f'DRUG-TREATED (from Section 6, Fisher-z-averaged across drugs): rho={np.tanh(z.mean()):.4f}')
print(f'Estevam et al. published drug-treated correlation for ESM-1b: r≈0.28')
print()
gap = untreated_rho - np.tanh(z.mean())
print(f'Untreated − treated gap (our reimplementation): {gap:.4f}')
print(f'Published untreated − treated gap: 0.50 − 0.28 = 0.22')
print()
if gap > 0.05:
    print('Direction and rough magnitude of the gap reproduce — consistent with the general-constraint-vs-drug-specific-pressure hypothesis.')
else:
    print('Gap is smaller than expected — investigate before treating this as a confirmed pattern.')

## 8. Non-learned baselines — BLOSUM62 + Estevam et al.'s own structural/biophysical features

Two structural distance baselines, kept separate as requested rather than one vague "structural distance": distance from the mutated residue to the **ATP-binding pocket** (`distance` column) and distance to the **bound inhibitor** (`inhib_distance` column) — both already computed by Estevam et al. and reused here, not recomputed. BLOSUM62 is computed fresh via Biopython.

**Not included here: sequence conservation.** That requires a multiple sequence alignment against MET homologs (e.g., via an HHblits/JackHMMER search against UniRef), which is a separate, heavier step — deliberately left as a follow-up rather than approximated with a placeholder number.

In [ ]:
blosum62 = substitution_matrices.load('BLOSUM62')

def blosum_score(row):
    try:
        return blosum62[row['wt_aa'], row['mut_aa']]
    except KeyError:
        return np.nan

mut_panel['blosum62'] = mut_panel.apply(blosum_score, axis=1)

merged_baselines = feat_df.merge(
    mut_panel[['pos_mut', 'our_esm1b_score', 'blosum62']], on='pos_mut', how='left'
)

baseline_score_cols = ['our_esm1b_score', 'blosum62', 'distance', 'inhib_distance',
                        'dddG', 'ddG_all', 'pocket_volume', 'hydrophobicity_score', 'polarity_score']

print('Pooled Spearman rho vs. drug-treated fitness, by baseline (for reference — report per-drug in the actual analysis):')
baseline_rows = []
for col in baseline_score_cols:
    sub = merged_baselines.dropna(subset=[col, 'mean'])
    rho, p = stats.spearmanr(sub[col], sub['mean'])
    baseline_rows.append({'feature': col, 'n': len(sub), 'rho_vs_treated': rho})
    print(f'  {col:22s}  n={len(sub):6d}  rho={rho:+.4f}')

baseline_df = pd.DataFrame(baseline_rows)
baseline_df.to_csv('met_baseline_comparison.csv', index=False)
print('\\nSaved met_baseline_comparison.csv')
print('\\nKey question this table answers: does ESM-1b beat the distance-to-pocket / distance-to-inhibitor')
print('baselines, or does a simple geometric feature already capture most of the signal?')

## 9. ESM-2, multiple sizes (150M / 650M / 3B) — the model-scale comparison

Only run after the Section 5 gate has passed. Run 150M and 650M first (fast); only run 3B if GPU memory allows (~12GB+ free) — it is the slowest and most memory-hungry step in this notebook.

In [ ]:
assert gate_rho >= GATE_THRESHOLD, 'Section 5 gate has not passed — fix the ESM-1b reimplementation before running ESM-2 sizes.'

esm2_checkpoints = {
    'esm2_150M': esm.pretrained.esm2_t30_150M_UR50D,
    'esm2_650M': esm.pretrained.esm2_t33_650M_UR50D,
    'esm2_3B':   esm.pretrained.esm2_t36_3B_UR50D,
}

all_scores = mut_panel[['pos_mut', 'pos', 'wt_aa', 'mut_aa', 'score', 'our_esm1b_score', 'blosum62'] + baseline_cols].copy()
all_scores = all_scores.rename(columns={'score': 'estevam_published_esm1b_score'})

for name, loader in esm2_checkpoints.items():
    print(f'\\nLoading {name} ...')
    try:
        model, alphabet = loader()
        model = model.to(device)
        logprobs = compute_masked_marginal_scores(model, alphabet, WT_SEQ, unique_positions, device)
        all_scores[f'{name}_score'] = all_scores.apply(
            lambda r: logprobs[r['pos']][r['mut_aa']] - logprobs[r['pos']][r['wt_aa']], axis=1
        )
        del model
        torch.cuda.empty_cache()
        print(f'{name} done.')
    except RuntimeError as e:
        print(f'{name} FAILED (likely out of GPU memory): {e}')
        print('Skip and continue - rerun this checkpoint alone on a larger GPU if needed.')

all_scores.to_csv('met_all_esm_scores.csv', index=False)
print('\\nSaved met_all_esm_scores.csv')
all_scores.head()

In [ ]:
# FIX: two bugs in the naive version of this cell.
# (1) 'hydrophobicity_score' and 'polarity_score' are baseline column names that happen to end in
#     '_score' too, so a naive `.endswith('_score')` filter wrongly swept them in as if they were
#     ESM model columns. Fixed by listing the real model columns explicitly instead of pattern-matching.
# (2) feat_df and all_scores both already carry the baseline columns (pos, dddG, distance, etc.), so a
#     merge on the full all_scores frame collided on those names and pandas silently suffixed them
#     _x/_y - which is why 'hydrophobicity_score' wasn't found. Fixed by merging in only pos_mut + the
#     actual model score columns, nothing that already exists in feat_df.

model_score_cols = ['our_esm1b_score'] + [f'{name}_score' for name in esm2_checkpoints if f'{name}_score' in all_scores.columns]
print('Model score columns being compared:', model_score_cols)

merged_all = feat_df.merge(all_scores[['pos_mut'] + model_score_cols], on='pos_mut', how='left')

print('\nrho(MET, drug, model) per drug per ESM checkpoint — full table, not pooled:')
full_results = []
for col in model_score_cols:
    for drug in sorted(merged_all['key'].unique()):
        sub = merged_all[merged_all['key'] == drug].dropna(subset=[col, 'mean'])
        if len(sub) == 0:
            continue
        rho, p = stats.spearmanr(sub[col], sub['mean'])
        full_results.append({'kinase': 'MET', 'drug': drug, 'model': col.replace('_score',''), 'n': len(sub), 'rho': rho})

full_results_df = pd.DataFrame(full_results)
full_results_df.to_csv('met_full_model_drug_correlation_table.csv', index=False)

print('\nHierarchical (across-drug) summary per model:')
for model_name, grp in full_results_df.groupby('model'):
    z = np.arctanh(grp['rho'].values)
    print(f'  {model_name:12s}  mean={grp["rho"].mean():.3f}  median={grp["rho"].median():.3f}  '
          f'fisher-z-avg={np.tanh(z.mean()):.3f}  range=[{grp["rho"].min():.3f}, {grp["rho"].max():.3f}]')

print('\nThis table (met_full_model_drug_correlation_table.csv) is the core evidence for whether ESM-2 scale')
print('beats ESM-1b, and whether either beats the Section 8 baselines, for MET drug-treated resistance.')
print('The same pipeline (Sections 4, 8, 9) should be re-run unchanged on EGFR and ABL1 sequences/mutation')
print('panels once the paired (matched WT-mutant, same-compound) ChEMBL data is ready.')

## Next steps (outside this notebook)

1. Confirm Section 5's gate printed PASSED before trusting anything below it.
2. Once passed, reuse `compute_masked_marginal_scores()` unchanged for EGFR (T790M/C797S/L858R, background-aware for the compound mutation) and ABL1 (M244V/G250E/Y253F/H/E255K/V/T315I/M351T/F359V/I) after pulling **matched, same-compound WT-mutant ChEMBL pairs** (the controlled &#916;pIC50 benchmark — not just the broader mutant-annotated record counts).
3. Compute the same BLOSUM62 + distance-to-pocket/distance-to-inhibitor baselines for EGFR/ABL1; distance features will need to be computed from a PDB structure (ABL1-imatinib: PDB 1IEP) since ChEMBL doesn't provide them the way Estevam et al.'s data already does for MET.
4. Add sequence conservation as a third baseline (needs an MSA against homologs — not done here).
5. Bootstrap confidence intervals + paired statistical comparison between model sizes, using `met_full_model_drug_correlation_table.csv` as the template for the EGFR/ABL1 equivalent tables.
6. Apply Benjamini-Hochberg FDR correction once the full model x kinase x drug x baseline comparison table exists across all three kinases.